# 04 · Grad-CAM: where does the model look?

**Goal:** a heatmap for every prediction, so a wrong answer is easy to spot.

**One trick that keeps serving simple:** our models end with global average pooling + one linear layer. For that head, Grad-CAM's gradient weights are exactly the classifier weights (divided by the map size). So Grad-CAM = **CAM = classifier weights × last feature maps**, no backward pass needed. The ONNX model only has to output the feature maps, and the API can draw the heatmap without PyTorch. Section 3 checks this against real autograd Grad-CAM.

The heatmap follows the final classifier: B0 + B3 ensemble, each with the flipped image (TTA), maps averaged.

Runs on CPU, Apple GPU or Kaggle GPU. **Inputs:** dataset, `splits.csv`, `classifier_b0_baseline.pt` + `classifier_b3_strong.pt`.

In [ ]:
import os
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image


def find_data_dir() -> Path:
    if "NEUROLENS_DATA" in os.environ:
        return Path(os.environ["NEUROLENS_DATA"])
    for p in Path("/kaggle/input").glob("**/Training"):
        return p.parent
    raise FileNotFoundError("Dataset not found. Attach masoudnickparvar/brain-tumor-mri-dataset.")


def find_file(name: str, local: Path) -> Path | None:
    for p in [*Path("/kaggle/input").glob(f"**/{name}"), local]:
        if p.exists():
            return p
    return None


DATA_DIR = find_data_dir()
SPLITS_CSV = Path(os.environ["NEUROLENS_SPLITS"]) if "NEUROLENS_SPLITS" in os.environ else find_file("splits.csv", Path("../data/splits.csv"))
MODELS_DIR = Path(os.environ.get("NEUROLENS_MODELS", "../models"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "b0": ("efficientnet_b0", 224, "classifier_b0_baseline.pt"),
    "b3": ("efficientnet_b3", 384, "classifier_b3_strong.pt"),
}
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

models = {}
for name, (arch, size, file) in MODELS.items():
    m = timm.create_model(arch, pretrained=False, num_classes=len(CLASSES))
    m.load_state_dict(torch.load(find_file(file, MODELS_DIR / file), map_location="cpu"))
    tf = A.Compose([A.Resize(size, size), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
    models[name] = (m.eval().to(device), tf)

df = pd.read_csv(SPLITS_CSV)
val_df = df[df.split == "val"].reset_index(drop=True)
print("device:", device, "| models:", list(models), "| val images:", len(val_df))

## 1. CAM from the classifier weights

In [ ]:
@torch.no_grad()
def probs_and_cams(model, x):
    """Class probabilities and one CAM per class: classifier weights x last feature maps."""
    feats = model.forward_features(x)                     # [B, C, h, w]
    probs = model.forward_head(feats).float().softmax(1)  # pooling + linear layer
    weights = model.get_classifier().weight               # [classes, C]
    cams = torch.einsum("kc,bchw->bkhw", weights, feats)  # [B, classes, h, w]
    return probs, cams


def explain(path) -> tuple[np.ndarray, int, np.ndarray, np.ndarray]:
    """Ensemble + TTA prediction and a heatmap (0..1, image size) for the predicted class."""
    img = np.array(Image.open(path).convert("RGB"))
    all_probs, all_cams = [], []
    for model, tf in models.values():
        x = tf(image=img)["image"][None].to(device)
        p1, c1 = probs_and_cams(model, x)
        p2, c2 = probs_and_cams(model, x.flip(3))
        all_probs.append((p1 + p2) / 2)
        all_cams.append((c1 + c2.flip(3)) / 2)            # flip the map back before averaging
    probs = torch.stack(all_probs).mean(0)[0].cpu().numpy()
    k = int(probs.argmax())

    heat = np.zeros(img.shape[:2], dtype=np.float32)
    for cams in all_cams:
        m = F.relu(cams[:, k:k + 1].float())              # keep only evidence *for* the class
        m = F.interpolate(m, size=img.shape[:2], mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
        heat += m / (m.max() + 1e-8) / len(all_cams)      # each model's map scaled to 0..1, then averaged
    return probs, k, heat, img

## 2. Predict every val image

In [ ]:
rows = []
for i, r in val_df.iterrows():
    probs, k, _, _ = explain(DATA_DIR / r.path)
    rows.append(dict(i=i, true=r.label, pred=CLASSES[k], conf=float(probs[k])))
pred_df = pd.DataFrame(rows)
print(f"ensemble+TTA val accuracy: {(pred_df.true == pred_df.pred).mean():.4f} | errors: {(pred_df.true != pred_df.pred).sum()}")

## 3. Check: CAM = Grad-CAM for this head

In [ ]:
def gradcam_autograd(model, x, k):
    """Classic Grad-CAM with a backward pass, only to check the shortcut above."""
    feats = model.forward_features(x).detach().requires_grad_(True)
    model.forward_head(feats)[0, k].backward()
    w = feats.grad.mean(dim=(2, 3), keepdim=True)         # gradient averaged over the map
    return F.relu((w * feats).sum(1))[0].detach()


for name, (model, tf) in models.items():
    for path in val_df.path[:3]:
        x = tf(image=np.array(Image.open(DATA_DIR / path).convert("RGB")))["image"][None].to(device)
        probs, cams = probs_and_cams(model, x)
        k = int(probs.argmax())
        a = F.relu(cams[0, k]).flatten().cpu()
        b = gradcam_autograd(model, x, k).flatten().cpu()
        corr = np.corrcoef(a.numpy(), b.numpy())[0, 1]
        print(f"{name} {Path(path).name}: correlation CAM vs Grad-CAM = {corr:.6f}")

## 4. Heatmaps per class (correct predictions)

In [ ]:
def overlay(ax, img, heat, title):
    ax.imshow(img[..., 0], cmap="gray")
    ax.imshow(heat, cmap="jet", alpha=0.4 * (heat > 0.2))  # hide the weakest part so the scan stays readable
    ax.set_title(title, fontsize=9)
    ax.axis("off")


rng = np.random.default_rng(0)
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(14, 14))
for r, c in enumerate(CLASSES):
    ok = pred_df[(pred_df.true == c) & (pred_df.pred == c)]
    for ax, i in zip(axes[r], rng.choice(ok.i.to_numpy(), 4, replace=False)):
        probs, k, heat, img = explain(DATA_DIR / val_df.path[i])
        overlay(ax, img, heat, f"{c} · {probs[k]:.0%}")
plt.tight_layout()
plt.savefig(OUT_DIR / "gradcam_per_class.png", dpi=70)
plt.show()

## 5. One example per class: scan and heatmap side by side (for the README)

In [ ]:
fig, axes = plt.subplots(2, len(CLASSES), figsize=(16, 8))
for col, c in enumerate(CLASSES):
    ok = pred_df[(pred_df.true == c) & (pred_df.pred == c)].sort_values("conf", ascending=False)
    i = ok.i.iloc[len(ok) // 4]                          # a typical, confident case (not the single easiest one)
    probs, k, heat, img = explain(DATA_DIR / val_df.path[i])
    axes[0, col].imshow(img[..., 0], cmap="gray")
    axes[0, col].set_title(c, fontsize=12)
    axes[0, col].axis("off")
    overlay(axes[1, col], img, heat, f"predicted {CLASSES[k]} · {probs[k]:.0%}")
plt.tight_layout()
plt.savefig(OUT_DIR / "gradcam_examples.png", dpi=80)
plt.show()

## 6. Heatmaps on the mistakes

In [ ]:
wrong = pred_df[pred_df.true != pred_df.pred].sort_values("conf", ascending=False)
if len(wrong):
    n = len(wrong)
    fig, axes = plt.subplots((n + 3) // 4, 4, figsize=(14, 3.6 * ((n + 3) // 4)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axes.flat, wrong.iterrows()):
        probs, k, heat, img = explain(DATA_DIR / val_df.path[r.i])
        overlay(ax, img, heat, f"true {r.true} · pred {r.pred} {r.conf:.0%}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "gradcam_errors.png", dpi=70)
    plt.show()